# Weed detection

This workflow prepares the supplied COCO dataset for YOLO detection. It does not download data, invent labels, or claim benchmark results. Set `WEED_DATA_ROOT` before running; missing data/artifacts stop with an actionable error.


In [ ]:
import os
from pathlib import Path

from weed_detection import (
    coco_to_yolo,
    require_directory,
    set_seed,
    split_dataset,
    write_dataset_yaml,
)

DATA_ROOT = Path(os.environ.get("WEED_DATA_ROOT", "data/weed-detection")).expanduser()
ARTIFACT_ROOT = Path(os.environ.get("WEED_ARTIFACT_ROOT", "artifacts")).expanduser()
SEED = int(os.environ.get("WEED_SEED", "42"))
TRAIN_ROOT = DATA_ROOT / "train"
ANNOTATIONS = TRAIN_ROOT / "_annotations.coco.json"
set_seed(SEED)
require_directory(TRAIN_ROOT, "weed-detection train directory")
if not ANNOTATIONS.is_file():
    raise FileNotFoundError(
        f"COCO annotations were not found: {ANNOTATIONS}. "
        "Set WEED_DATA_ROOT to a directory containing train/_annotations.coco.json."
    )
print(f"Input: {TRAIN_ROOT.resolve()}")
print(f"Artifacts: {ARTIFACT_ROOT.resolve()}")
print(f"Seed: {SEED}")


In [ ]:
CONVERTED_ROOT = ARTIFACT_ROOT / "converted"
conversion_report = coco_to_yolo(ANNOTATIONS, TRAIN_ROOT, CONVERTED_ROOT)
print(conversion_report)


## Leakage-safe split

The split is performed before any offline augmentation. Roboflow-style `.rf.` filename derivatives are kept in one group. The training framework may apply online augmentation only to training examples.


In [ ]:
DATASET_ROOT = ARTIFACT_ROOT / "dataset"
split_report = split_dataset(
    CONVERTED_ROOT / "images",
    DATASET_ROOT,
    validation_fraction=float(os.environ.get("WEED_VALIDATION_FRACTION", "0.2")),
    seed=SEED,
)
dataset_yaml = write_dataset_yaml(
    DATASET_ROOT, conversion_report["class_names"], ARTIFACT_ROOT / "dataset.yaml"
)
print(split_report)
print(f"Dataset config: {dataset_yaml}")


## Optional training

Training is disabled unless `WEED_RUN_TRAINING=1`. Install `requirements.txt`, supply the real dataset, and ensure model-weight download permissions or set `WEED_MODEL_WEIGHTS` to a local weights file.


In [ ]:
model = None
run_training = os.environ.get("WEED_RUN_TRAINING", "0") == "1"
if run_training:
    try:
        from ultralytics import YOLO
    except ImportError as exc:
        raise RuntimeError("Training requires ultralytics; install requirements.txt first.") from exc
    weights = os.environ.get("WEED_MODEL_WEIGHTS", "yolov8s.pt")
    model = YOLO(weights)
    model.train(
        data=str(dataset_yaml),
        epochs=int(os.environ.get("WEED_EPOCHS", "170")),
        imgsz=int(os.environ.get("WEED_IMAGE_SIZE", "640")),
        batch=int(os.environ.get("WEED_BATCH_SIZE", "16")),
        name="weed_detection",
        seed=SEED,
        deterministic=True,
        project=str(ARTIFACT_ROOT / "runs"),
    )
else:
    print("Training skipped. Set WEED_RUN_TRAINING=1 only when the dataset and dependencies are available.")


## Validation metrics

Metrics are read from a real model run only. They are validation metrics, not an independent test benchmark. No metric is produced when the model artifact is absent.


In [ ]:
if model is None:
    try:
        from ultralytics import YOLO
    except ImportError:
        YOLO = None
    candidate = Path(os.environ.get("WEED_MODEL_PATH", str(ARTIFACT_ROOT / "runs" / "weed_detection" / "weights" / "best.pt")))
    if YOLO is not None and candidate.is_file():
        model = YOLO(str(candidate))
    else:
        print("Validation skipped: no model artifact is available.")

metrics = None
if model is not None:
    metrics = model.val(data=str(dataset_yaml), split="val", plots=False, seed=SEED)
    for key in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"):
        print(f"{key}: {metrics.results_dict.get(key, 'unavailable')}")


## Optional single-image inference

Set `WEED_IMAGE` to an existing image and provide a model. This cell never falls back to a repository-specific filename.


In [ ]:
image_value = os.environ.get("WEED_IMAGE")
if model is None:
    print("Inference skipped: no model artifact is available.")
elif not image_value:
    print("Inference skipped: set WEED_IMAGE to an existing image path.")
else:
    image_path = Path(image_value).expanduser().resolve()
    if not image_path.is_file():
        raise FileNotFoundError(f"Inference image was not found: {image_path}")
    predictions = model.predict(source=str(image_path), save=False)
    output_path = ARTIFACT_ROOT / "predictions" / f"{image_path.stem}_predicted.jpg"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    predictions[0].save(filename=str(output_path))
    print(f"Prediction saved to: {output_path}")


## Optional training-history plot

This reads a real `results.csv` from a run and skips cleanly when it is absent.


In [ ]:
results_csv = ARTIFACT_ROOT / "runs" / "weed_detection" / "results.csv"
if not results_csv.is_file():
    print(f"History plot skipped: artifact not found: {results_csv}")
else:
    try:
        import matplotlib.pyplot as plt
        import pandas as pd
    except ImportError as exc:
        raise RuntimeError("Plotting requires pandas and matplotlib; install requirements.txt.") from exc
    history = pd.read_csv(results_csv)
    metric_columns = [column for column in ("metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)") if column in history]
    if not metric_columns:
        raise ValueError(f"No supported metric columns found in {results_csv}")
    history.plot(x="epoch", y=metric_columns, figsize=(10, 5), grid=True, title="Validation metrics")
    plt.tight_layout()
    plt.show()
